In [ ]:
import glob
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

In [ ]:
HERE = os.getcwd()
SEED = int(os.environ.get("SEED", 0))
RES = os.path.join(HERE, "..", "results" if SEED == 0 else f"results_seed{SEED}")
FIG = os.path.join(HERE, "..", "figures")
DATA = os.path.join(HERE, "..", "data")
for folder in (RES, FIG, DATA):
    os.makedirs(folder, exist_ok=True)

WINDOWS = [6, 12, 24, 48]
FIXED_H = 24
FA_BUDGET = 0.10
COVERAGE_FLOOR = 0.80
N_BOOT = 1000
EPS = 1e-6
MODELS = ["lr", "lr_miss", "grud"]
LABEL = {"lr": "Logistic regression", "lr_miss": "LR, missingness only", "grud": "GRU-D"}


def load_data():
    path = os.path.join(DATA, f"physionet2012_seed{SEED}.npz")
    if not os.path.exists(path):
        from benchpots.datasets import preprocess_physionet2012
        d = preprocess_physionet2012(subset="all", rate=0, task_type="classification", random_state=SEED)
        np.savez_compressed(
            path, Xtr=d["train_X"], Xva=d["val_X"], Xte=d["test_X"],
            ytr=d["train_y"].astype(int), yva=d["val_y"].astype(int), yte=d["test_y"].astype(int),
            icu_te=d["test_ICUType"])
    with np.load(path) as z:
        return {k: z[k] for k in z.files}


def window(X, h):
    X = X.copy()
    X[:, h:, :] = np.nan
    return X


def lr_feats(X, missingness_only=False):
    obs = ~np.isnan(X)
    ind = obs.any(1).astype(float)
    cnt = np.log1p(obs.sum(1))
    if missingness_only:
        return np.concatenate([ind, cnt], 1)
    t_idx = np.where(obs, np.arange(X.shape[1])[None, :, None], -1).max(1)
    last = np.take_along_axis(np.nan_to_num(X), t_idx.clip(0)[:, None, :], 1)[:, 0, :]
    last[t_idx < 0] = 0.0
    return np.concatenate([last, ind, cnt], 1)


def logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def ece(p, y, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    return float(sum(abs(p[idx == b].mean() - y[idx == b].mean()) * (idx == b).mean()
                     for b in range(bins) if (idx == b).any()))


def run_fixed(p_fixed, thr):
    flag = p_fixed >= thr
    n = len(p_fixed)
    return flag, np.full(n, float(FIXED_H)), np.ones(n, bool)


def run_halting(Pm, tlo, thi):
    n = Pm.shape[0]
    dec = np.zeros(n, int)
    hour = np.full(n, float(WINDOWS[-1]))
    for j, h in enumerate(WINDOWS):
        open_ = dec == 0
        hi = open_ & (Pm[:, j] >= thi)
        lo = open_ & (Pm[:, j] <= tlo)
        dec[hi], dec[lo] = 1, -1
        hour[hi | lo] = h
    return dec == 1, hour, dec != 0


def policy_metrics(flag, hour, decided, y):
    pos, neg = y == 1, y == 0
    cov = decided.mean()
    return dict(
        sens=float(flag[pos].mean()),
        fpr=float(flag[neg].mean()),
        coverage=float(cov),
        mean_hour=float(hour.mean()),
        err_covered=float((flag[decided] != y[decided]).mean()) if cov > 0 else float("nan"),
    )

In [ ]:
d = load_data()
Xtr, Xva, Xte = d["Xtr"], d["Xva"], d["Xte"]
summary = dict(
    n=dict(train=int(len(d["ytr"])), val=int(len(d["yva"])), test=int(len(d["yte"])),
           total=int(len(d["ytr"]) + len(d["yva"]) + len(d["yte"]))),
    n_steps=int(Xtr.shape[1]), n_features=int(Xtr.shape[2]),
    prevalence=dict(train=float(d["ytr"].mean()), val=float(d["yva"].mean()), test=float(d["yte"].mean())),
    missing_rate_train=float(np.isnan(Xtr).mean()),
    observed_values_per_test_patient={
        str(h): dict(mean=float((~np.isnan(window(Xte, h))).sum((1, 2)).mean()),
                     median=float(np.median((~np.isnan(window(Xte, h))).sum((1, 2)))))
        for h in WINDOWS},
    icu_type_share_test={str(int(k)): float(v) for k, v in
                         zip(*np.unique(d["icu_te"], return_counts=True)) for v in [v / len(d["icu_te"])]},
)
with open(os.path.join(RES, "data_summary.json"), "w") as fh:
    json.dump(summary, fh, indent=1)
print(json.dumps(summary, indent=1))

In [ ]:
NO_GRUD = False
EPOCHS = 40
np.random.seed(SEED)
torch.manual_seed(SEED)

d = load_data()
Xtr, Xva, Xte, ytr, yva, yte = d["Xtr"], d["Xva"], d["Xte"], d["ytr"], d["yva"], d["yte"]
N, T, F = Xtr.shape
models = [m for m in MODELS if not (NO_GRUD and m == "grud")]


def fit_lr(Xa, ya, Xb_list, missingness_only=False):
    clf = LogisticRegression(C=0.1, max_iter=2000)
    clf.fit(lr_feats(Xa, missingness_only), ya)
    return [clf.predict_proba(lr_feats(Xb, missingness_only))[:, 1] for Xb in Xb_list]


def fit_grud(Xa, ya, Xv, yv, Xb_list):
    from pypots.classification import GRUD
    from pypots.utils.random import set_random_seed
    set_random_seed(SEED)
    m = GRUD(n_steps=T, n_features=F, n_classes=2, rnn_hidden_size=64, batch_size=64,
             epochs=EPOCHS, patience=5, device="cpu", model_saving_strategy=None, verbose=False)
    m.fit({"X": Xa, "y": ya}, {"X": Xv, "y": yv})
    return [m.predict({"X": Xb})["classification_proba"][:, 1] for Xb in Xb_list]


out = {"yva": yva, "yte": yte, "icu_te": d["icu_te"]}
for m in models:
    out[f"{m}_val"] = np.zeros((len(yva), len(WINDOWS)))
    out[f"{m}_test"] = np.zeros((len(yte), len(WINDOWS)))

for j, h in enumerate(WINDOWS):
    Xtr_h, Xva_h, Xte_h = window(Xtr, h), window(Xva, h), window(Xte, h)
    out["lr_val"][:, j], out["lr_test"][:, j] = fit_lr(Xtr_h, ytr, [Xva_h, Xte_h])
    out["lr_miss_val"][:, j], out["lr_miss_test"][:, j] = fit_lr(Xtr_h, ytr, [Xva_h, Xte_h], missingness_only=True)
    if "grud" in models:
        out["grud_val"][:, j], out["grud_test"][:, j] = fit_grud(Xtr_h, ytr, Xva_h, yva, [Xva_h, Xte_h])
    print(f"h={h:2d}: " + "  ".join(f"{m} test AUROC={roc_auc_score(yte, out[f'{m}_test'][:, j]):.3f}" for m in models), flush=True)

np.savez(os.path.join(RES, "probs.npz"), **out)
print("saved", os.path.join(RES, "probs.npz"))

In [ ]:
rng = np.random.default_rng(SEED)
Z = np.load(os.path.join(RES, "probs.npz"))
yva, yte = Z["yva"], Z["yte"]
MODELS = sorted({k.rsplit("_", 1)[0] for k in Z.files if k.endswith("_val")})
P = {m: {"val": Z[f"{m}_val"], "test": Z[f"{m}_test"]} for m in MODELS}
j24 = WINDOWS.index(FIXED_H)


def platt(p_val, y_val):
    c = LogisticRegression(C=1e6, max_iter=2000).fit(logit(p_val)[:, None], y_val)
    return float(c.coef_[0, 0]), float(c.intercept_[0])


PC, RECAL = {}, {}
for m in MODELS:
    PC[m] = {"val": np.zeros_like(P[m]["val"]), "test": np.zeros_like(P[m]["test"])}
    RECAL[m] = []
    for j in range(len(WINDOWS)):
        a, b = platt(P[m]["val"][:, j], yva)
        RECAL[m].append([a, b])
        for s in ("val", "test"):
            PC[m][s][:, j] = sigmoid(a * logit(P[m][s][:, j]) + b)


def tune_fixed(p_val, y_val):
    return float(np.quantile(p_val[y_val == 0], 1 - FA_BUDGET))


def tune_halting(Pv, y_val):
    qs = np.quantile(Pv, np.linspace(0.01, 0.99, 99))
    best, best_key = None, None
    for thi in qs:
        for tlo in qs:
            if tlo >= thi:
                continue
            mt = policy_metrics(*run_halting(Pv, tlo, thi), y_val)
            if mt["fpr"] > FA_BUDGET or mt["coverage"] < COVERAGE_FLOOR:
                continue
            key = (round(mt["sens"], 3), -round(mt["mean_hour"], 1))
            if best_key is None or key > best_key:
                best, best_key = (float(tlo), float(thi)), key
    return best


THR, RUNS = {}, {}
for m in MODELS:
    for cal, PP in (("raw", P), ("cal", PC)):
        thr_f = tune_fixed(PP[m]["val"][:, j24], yva)
        tlo, thi = tune_halting(PP[m]["val"], yva)
        THR[f"{m}|{cal}"] = dict(fixed=thr_f, tlo=tlo, thi=thi)
        RUNS[f"fixed|{m}|{cal}"] = run_fixed(PP[m]["test"][:, j24], thr_f)
        RUNS[f"halting|{m}|{cal}"] = run_halting(PP[m]["test"], tlo, thi)
        print(f"{m}/{cal}: fixed thr={thr_f:.3f}  halting tlo={tlo:.3f} thi={thi:.3f}")

if "grud" in MODELS and "lr" in MODELS:
    for src, dst in (("lr", "grud"), ("grud", "lr")):
        for cal, PP in (("raw", P), ("cal", PC)):
            t = THR[f"{src}|{cal}"]
            RUNS[f"fixed|{dst}<-{src}|{cal}"] = run_fixed(PP[dst]["test"][:, j24], t["fixed"])
            RUNS[f"halting|{dst}<-{src}|{cal}"] = run_halting(PP[dst]["test"], t["tlo"], t["thi"])


def boot_ci(fn, n=N_BOOT):
    vals = np.array([fn(rng.integers(0, len(yte), len(yte))) for _ in range(n)])
    return [float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))]


results = dict(windows=WINDOWS, fixed_h=FIXED_H, fa_budget=FA_BUDGET, coverage_floor=COVERAGE_FLOOR,
               n_test=int(len(yte)), n_test_deaths=int(yte.sum()),
               recalibration=RECAL, thresholds=THR, discrimination={}, policies={}, tests={})

disc_rows = []
for m in MODELS:
    for j, h in enumerate(WINDOWS):
        pt, pct = P[m]["test"][:, j], PC[m]["test"][:, j]
        r = dict(auroc=float(roc_auc_score(yte, pt)), auroc_ci=boot_ci(lambda i: roc_auc_score(yte[i], pt[i])),
                 auprc=float(average_precision_score(yte, pt)),
                 auprc_ci=boot_ci(lambda i: average_precision_score(yte[i], pt[i])),
                 ece_raw=ece(pt, yte), ece_cal=ece(pct, yte))
        results["discrimination"][f"{m}|{h}"] = r
        disc_rows.append(dict(model=m, hours=h, **{k: v for k, v in r.items() if not k.endswith("_ci")},
                              auroc_lo=r["auroc_ci"][0], auroc_hi=r["auroc_ci"][1]))
pd.DataFrame(disc_rows).to_csv(os.path.join(RES, "table_discrimination.csv"), index=False)

if "grud" in MODELS and "lr" in MODELS:
    for j, h in enumerate(WINDOWS):
        a, b = P["grud"]["test"][:, j], P["lr"]["test"][:, j]
        results["tests"][f"auroc_diff_grud_minus_lr|{h}"] = dict(
            diff=float(roc_auc_score(yte, a) - roc_auc_score(yte, b)),
            ci=boot_ci(lambda i: roc_auc_score(yte[i], a[i]) - roc_auc_score(yte[i], b[i])))

pol_rows = []
for name, (f, hr, dcd) in RUNS.items():
    mt = policy_metrics(f, hr, dcd, yte)
    mt["sens_ci"] = boot_ci(lambda i: f[i][yte[i] == 1].mean())
    mt["fpr_ci"] = boot_ci(lambda i: f[i][yte[i] == 0].mean())
    mt["coverage_ci"] = boot_ci(lambda i: dcd[i].mean())
    mt["mean_hour_ci"] = boot_ci(lambda i: hr[i].mean())
    results["policies"][name] = mt
    pol_rows.append(dict(policy=name, **{k: v for k, v in mt.items() if not k.endswith("_ci")}))
pd.DataFrame(pol_rows).to_csv(os.path.join(RES, "table_policies.csv"), index=False)

for m in MODELS:
    for cal in ("raw", "cal"):
        f1, h1, _ = RUNS[f"fixed|{m}|{cal}"]
        f2, h2, _ = RUNS[f"halting|{m}|{cal}"]
        pos = yte == 1
        b = int((f1[pos] & ~f2[pos]).sum())
        c = int((~f1[pos] & f2[pos]).sum())
        results["tests"][f"fixed_vs_halting|{m}|{cal}"] = dict(
            fixed_only=b, halting_only=c,
            mcnemar_p=float(binomtest(min(b, c), b + c, 0.5).pvalue) if b + c else 1.0,
            hour_diff=float((h2 - h1).mean()), hour_diff_ci=boot_ci(lambda i: (h2[i] - h1[i]).mean()))

def sens_at_fpr(y, p, fpr):
    thr = np.quantile(p[y == 0], 1 - fpr)
    return (p[y == 1] >= thr).mean()


for m in MODELS:
    f_h = RUNS[f"halting|{m}|cal"][0]
    p24 = PC[m]["test"][:, j24]
    fpr_h = f_h[yte == 0].mean()
    sens_h = f_h[yte == 1].mean()
    sens_f = sens_at_fpr(yte, p24, fpr_h)

    def diff(i):
        y, f, p = yte[i], f_h[i], p24[i]
        return f[y == 1].mean() - sens_at_fpr(y, p, f[y == 0].mean())
    results["tests"][f"matched_fpr|{m}"] = dict(
        fpr=float(fpr_h), halting_sens=float(sens_h), fixed_sens_at_same_fpr=float(sens_f),
        diff=float(sens_h - sens_f), diff_ci=boot_ci(diff))

dec_out = {"main_model": "grud" if "grud" in MODELS else "lr"}
for m in MODELS:
    f, hr, dcd = RUNS[f"halting|{m}|cal"]
    dec_out.update({f"{m}_flag": f, f"{m}_hour": hr, f"{m}_decided": dcd, f"{m}_p_cal": PC[m]["test"],
                    f"{m}_thi": THR[f"{m}|cal"]["thi"], f"{m}_tlo": THR[f"{m}|cal"]["tlo"]})
np.savez(os.path.join(RES, "decisions.npz"), **dec_out)

with open(os.path.join(RES, "results.json"), "w") as fh:
    json.dump(results, fh, indent=1)
print(pd.DataFrame(pol_rows).round(3).to_string(index=False))
print("saved results.json")

In [ ]:
r = json.load(open(os.path.join(RES, "results.json")))
Z = np.load(os.path.join(RES, "probs.npz"))
yte = Z["yte"]
models = sorted({k.split("|")[0] for k in r["discrimination"]}, key=["lr", "lr_miss", "grud"].index)
j24 = WINDOWS.index(FIXED_H)

fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.6))
for ax, met in zip(axes, ("auroc", "auprc")):
    for m in models:
        v = np.array([r["discrimination"][f"{m}|{h}"][met] for h in WINDOWS])
        lo = np.array([r["discrimination"][f"{m}|{h}"][met + "_ci"][0] for h in WINDOWS])
        hi = np.array([r["discrimination"][f"{m}|{h}"][met + "_ci"][1] for h in WINDOWS])
        ax.errorbar(WINDOWS, v, yerr=[v - lo, hi - v], marker="o", ms=3, capsize=2, label=LABEL[m])
    ax.set_xticks(WINDOWS)
    ax.set_xlabel("Hours observed")
    ax.set_ylabel(met.upper())
    ax.grid(alpha=.3)
axes[0].axhline(0.5, color="grey", lw=.8, ls=":")
axes[1].axhline(yte.mean(), color="grey", lw=.8, ls=":")
axes[0].legend(fontsize=7, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIG, "fig1_earliness.pdf"))

yva = Z["yva"]
D = np.load(os.path.join(RES, "decisions.npz"))
fig, ax = plt.subplots(figsize=(3.2, 2.6))
for m in [x for x in models if x != "lr_miss"]:
    color = None
    for jj, ls in ((j24, "-"), (len(WINDOWS) - 1, "--")):
        a, b = r["recalibration"][m][jj]
        p = sigmoid(a * logit(Z[f"{m}_test"][:, jj]) + b)
        pv = sigmoid(a * logit(Z[f"{m}_val"][:, jj]) + b)
        thr = float(np.quantile(pv[yva == 0], 0.9))
        dist = np.abs(logit(p) - logit(thr))
        cov, err = [], []
        for delta in np.quantile(dist, np.linspace(0, 0.95, 40)):
            keep = dist >= delta
            yy, fl = yte[keep], p[keep] >= thr
            if (yy == 1).sum() == 0 or (yy == 0).sum() == 0:
                continue
            cov.append(keep.mean())
            err.append(0.5 * ((~fl[yy == 1]).mean() + fl[yy == 0].mean()))
        line, = ax.plot(cov, err, ls, ms=3, color=color, label=f"{LABEL[m]}, {WINDOWS[jj]} h")
        color = line.get_color()
    dcd, fl = D[f"{m}_decided"], D[f"{m}_flag"]
    yd, fd = yte[dcd], fl[dcd]
    ax.scatter([dcd.mean()], [0.5 * ((~fd[yd == 1]).mean() + fd[yd == 0].mean())],
               marker="*", s=70, zorder=5, color=color)
ax.set_xlabel("Coverage (fraction of patients decided)")
ax.set_ylabel("Balanced error among decided")
ax.grid(alpha=.3)
ax.legend(fontsize=7, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIG, "fig2_risk_coverage.pdf"))
print("figures written to", FIG)

In [ ]:
D = np.load(os.path.join(RES, "decisions.npz"))
m = str(D["main_model"])
Z = np.load(os.path.join(RES, "probs.npz"))
yte, icu = Z["yte"], Z["icu_te"]
Xte = load_data()["Xte"]
thi, tlo = float(D[f"{m}_thi"]), float(D[f"{m}_tlo"])
print("halting rule, recalibrated", m)

df = pd.DataFrame({"y": yte, "icu_type": icu, "flag": D[f"{m}_flag"], "hour": D[f"{m}_hour"],
                   "decided": D[f"{m}_decided"],
                   "n_vars_48h": (~np.isnan(Xte)).any(1).sum(1),
                   **{f"n_obs_{h}h": (~np.isnan(window(Xte, h))).sum((1, 2)) for h in WINDOWS},
                   **{f"p_{h}": D[f"{m}_p_cal"][:, j] for j, h in enumerate(WINDOWS)}})
df["group"] = np.select(
    [~df.decided, df.flag & (df.y == 0), df.flag & (df.y == 1), ~df.flag & (df.y == 1)],
    ["abstain", "false_alarm", "true_alarm", "miss"], default="true_clear")
df.to_csv(os.path.join(RES, "cases.csv"), index=False)
g = df.groupby("group").agg(n=("y", "size"), death_rate=("y", "mean"), median_obs_48h=("n_obs_48h", "median"),
                            median_vars=("n_vars_48h", "median"), mean_hour=("hour", "mean"),
                            **{f"median_p{h}": (f"p_{h}", "median") for h in WINDOWS})
g.to_csv(os.path.join(RES, "case_groups.csv"))
print(g.round(3).to_string())

ab, fa, ta, ms = (df[df.group == k] for k in ("abstain", "false_alarm", "true_alarm", "miss"))
print(f"\nthresholds: clear <= {tlo:.3f}, flag >= {thi:.3f}")
print(f"decided by 6 h: {(df.hour <= 6).mean():.3f}; by 12 h: {(df.hour <= 12).mean():.3f}; "
      f"mean hour among decided: {df[df.decided].hour.mean():.1f}")
print(f"abstained: {len(ab)} ({len(ab)/len(df):.3f}); deaths among them {ab.y.mean():.3f}; "
      f"median obs 48h {ab.n_obs_48h.median()} vs decided {df[df.decided].n_obs_48h.median()}; "
      f"p48 IQR {ab.p_48.quantile(.25):.2f}-{ab.p_48.quantile(.75):.2f}")
print(f"false alarms: {len(fa)}; flagged at 6 h: {(fa.hour == 6).sum()}, of which p24 < flag thr: "
      f"{((fa.hour == 6) & (fa.p_24 < thi)).sum()}; p48 < flag thr: {(fa.p_48 < thi).sum()}")
print(f"MICU share: all {(df.icu_type == 3).mean():.2f}, false alarms {(fa.icu_type == 3).mean():.2f}; "
      f"CSRU share: all {(df.icu_type == 2).mean():.2f}, true clears {(df[df.group == 'true_clear'].icu_type == 2).mean():.2f}")
print(f"misses: {len(ms)}; cleared at 6 h: {(ms.hour == 6).sum()}; median p48 {ms.p_48.median():.3f}")

In [ ]:
runs = {0: os.path.join(HERE, "..", "results", "results.json")}
for p in sorted(glob.glob(os.path.join(HERE, "..", "results_seed*", "results.json"))):
    runs[int(os.path.basename(os.path.dirname(p)).replace("results_seed", ""))] = p
R = {s: json.load(open(p)) for s, p in runs.items()}

rows = {
    "AUROC 6 h, GRU-D": lambda r: r["discrimination"]["grud|6"]["auroc"],
    "AUROC 6 h, LR": lambda r: r["discrimination"]["lr|6"]["auroc"],
    "AUROC 48 h, GRU-D": lambda r: r["discrimination"]["grud|48"]["auroc"],
    "AUROC 48 h, LR": lambda r: r["discrimination"]["lr|48"]["auroc"],
    "AUROC 6 h, missingness only": lambda r: r["discrimination"]["lr_miss|6"]["auroc"],
    "fixed FPR, LR": lambda r: r["policies"]["fixed|lr|cal"]["fpr"],
    "fixed FPR, GRU-D": lambda r: r["policies"]["fixed|grud|cal"]["fpr"],
    "halting FPR, LR": lambda r: r["policies"]["halting|lr|cal"]["fpr"],
    "halting FPR, GRU-D": lambda r: r["policies"]["halting|grud|cal"]["fpr"],
    "halting coverage, GRU-D": lambda r: r["policies"]["halting|grud|cal"]["coverage"],
    "halting sens - fixed sens, GRU-D": lambda r: r["policies"]["halting|grud|cal"]["sens"] - r["policies"]["fixed|grud|cal"]["sens"],
    "halting sens - fixed sens, LR": lambda r: r["policies"]["halting|lr|cal"]["sens"] - r["policies"]["fixed|lr|cal"]["sens"],
    "matched-FPR sens diff, GRU-D": lambda r: r["tests"]["matched_fpr|grud"]["diff"],
    "matched-FPR sens diff, LR": lambda r: r["tests"]["matched_fpr|lr"]["diff"],
    "transfer LR->GRU-D coverage shift (raw)": lambda r: r["policies"]["halting|grud<-lr|raw"]["coverage"] - r["policies"]["halting|grud|raw"]["coverage"],
    "transfer LR->GRU-D coverage shift (recal)": lambda r: r["policies"]["halting|grud<-lr|cal"]["coverage"] - r["policies"]["halting|grud|cal"]["coverage"],
    "transfer GRU-D->LR coverage shift (raw)": lambda r: r["policies"]["halting|lr<-grud|raw"]["coverage"] - r["policies"]["halting|lr|raw"]["coverage"],
    "transfer GRU-D->LR coverage shift (recal)": lambda r: r["policies"]["halting|lr<-grud|cal"]["coverage"] - r["policies"]["halting|lr|cal"]["coverage"],
    "transfer LR->GRU-D FPR shift (raw)": lambda r: r["policies"]["halting|grud<-lr|raw"]["fpr"] - r["policies"]["halting|grud|raw"]["fpr"],
    "transfer GRU-D->LR FPR shift (raw)": lambda r: r["policies"]["halting|lr<-grud|raw"]["fpr"] - r["policies"]["halting|lr|raw"]["fpr"],
    "fixed 24 h FPR shift LR->GRU-D (raw)": lambda r: r["policies"]["fixed|grud<-lr|raw"]["fpr"] - r["policies"]["fixed|grud|raw"]["fpr"],
    "fixed 24 h FPR shift GRU-D->LR (raw)": lambda r: r["policies"]["fixed|lr<-grud|raw"]["fpr"] - r["policies"]["fixed|lr|raw"]["fpr"],
}
out = {}
print(f"{'quantity':45s}" + "".join(f"seed{s:<8d}" for s in R) + "  mean    min     max")
for name, fn in rows.items():
    v = np.array([fn(R[s]) for s in R])
    out[name] = dict(per_seed={str(s): float(x) for s, x in zip(R, v)}, mean=float(v.mean()), min=float(v.min()), max=float(v.max()))
    print(f"{name:45s}" + "".join(f"{x:<12.3f}" for x in v) + f"  {v.mean():.3f}  {v.min():.3f}  {v.max():.3f}")
with open(os.path.join(HERE, "..", "results", "seed_summary.json"), "w") as fh:
    json.dump(out, fh, indent=1)

In [ ]:
P = np.array([[0.9, 0.9, 0.9, 0.9],
              [0.05, 0.9, 0.9, 0.9],
              [0.5, 0.5, 0.95, 0.5],
              [0.5, 0.5, 0.5, 0.5]])
flag, hour, decided = run_halting(P, tlo=0.1, thi=0.8)
assert flag.tolist() == [True, False, True, False]
assert hour.tolist() == [6, 6, 24, 48]
assert decided.tolist() == [True, True, True, False]
m = policy_metrics(flag, hour, decided, np.array([1, 1, 0, 1]))
assert abs(m["sens"] - 1 / 3) < 1e-9 and m["fpr"] == 1.0 and m["coverage"] == 0.75
print("halting rule unit checks: ok")

r = json.load(open(os.path.join(RES, "results.json")))
Z = np.load(os.path.join(RES, "probs.npz"))
yte = Z["yte"]
for m_ in ("lr", "grud"):
    if f"{m_}_test" not in Z.files:
        continue
    for j, h in enumerate(WINDOWS):
        a = roc_auc_score(yte, Z[f"{m_}_test"][:, j])
        assert abs(a - r["discrimination"][f"{m_}|{h}"]["auroc"]) < 1e-9, (m_, h)
print("AUROC per window matches results.json: ok")

thr = r["thresholds"]["lr|raw"]["fixed"]
p24 = Z["lr_test"][:, WINDOWS.index(24)]
sens = (p24[yte == 1] >= thr).mean()
fpr = (p24[yte == 0] >= thr).mean()
pol = r["policies"]["fixed|lr|raw"]
assert abs(sens - pol["sens"]) < 1e-9 and abs(fpr - pol["fpr"]) < 1e-9
print(f"fixed rule (LR, raw) re-derived: sens={sens:.3f} fpr={fpr:.3f}: ok")

t = r["thresholds"]["lr|raw"]
Pm = Z["lr_test"]
flags, hours = [], []
for i in range(len(yte)):
    f, hr = False, 48.0
    for j, h in enumerate(WINDOWS):
        if Pm[i, j] >= t["thi"]:
            f, hr = True, h
            break
        if Pm[i, j] <= t["tlo"]:
            f, hr = False, h
            break
    flags.append(f)
    hours.append(hr)
flags, hours = np.array(flags), np.array(hours)
pol = r["policies"]["halting|lr|raw"]
assert abs(flags[yte == 1].mean() - pol["sens"]) < 1e-9
assert abs(flags[yte == 0].mean() - pol["fpr"]) < 1e-9
assert abs(hours.mean() - pol["mean_hour"]) < 1e-9
print(f"halting rule (LR, raw) re-derived: sens={flags[yte==1].mean():.3f} "
      f"fpr={flags[yte==0].mean():.3f} mean hour={hours.mean():.1f}: ok")
print("all checks passed")